# GATED — AI RECONSTRUCTIONS PERMITTED, GENERATION NOT AUTHORIZED

SonicVhibez permits clearly disclosed, photorealistic AI wildlife reconstructions. This notebook is **not authorized to generate yet**: every code cell checks the immutable policy file pinned below and stops unless a zero-cost GPU backend and the exact checkpoint/VAE rights and provenance have both been independently verified and explicitly enabled. The current policy keeps those gates closed. Do not bypass the checks, spend GPU credits, or treat generated test output as authentic wildlife footage. Nature Frontiers is a separate brand and is not governed by this notebook.

# SonicVhibez — project-owned WanGP on Kaggle

**Purpose:** one controlled 5-second photorealistic wildlife test shot. This notebook pins the WanGP source commit and the exact quantized Wan 2.2 5B checkpoint file. It does not publish or approve anything.

**Kaggle settings:** enable Internet and select GPU (T4 preferred; P100 fallback if offered). Free GPU availability and session limits apply. Model weights/cache are stored in `/kaggle/temp`; the small test output is saved in `/kaggle/working`.

**Security:** this notebook creates a temporary public Gradio share URL for the WanGP UI. Do not enter passwords, API keys, YouTube credentials or private material into the UI. Stop the launch cell and end the Kaggle session when finished.

**Commercial-use hold:** the official Wan-AI Wan2.2 TI2V-5B model card declares Apache-2.0. The exact WanGP-compatible single-file mirror checkpoint below has a pinned file hash, but its mirror repository does not independently declare a licence or explicit provenance statement for that converted file. The test is for internal evaluation only; commercial use stays blocked until the mirror's provenance/rights are confirmed. No FastWan LoRA is used.


In [ ]:
raise RuntimeError("DISABLED_BY_OWNER_AI_WILDLIFE_VISUAL_POLICY: original creator-captured wildlife media is required; AI wildlife generation is prohibited.")

import json, urllib.request

POLICY_COMMIT = "4a99cc9abfb47fcf8d32dc517711322ea8a08fff"
POLICY_URL = (
    "https://raw.githubusercontent.com/clickerrish-art/YT-AUTO-VISE/"
    + POLICY_COMMIT + "/config/video-generation.json"
)

def require_generation_authorized():
    try:
        with urllib.request.urlopen(POLICY_URL, timeout=20) as response:
            policy = json.loads(response.read().decode("utf-8"))
    except Exception as exc:
        raise RuntimeError("BLOCKED: unable to read the pinned SonicVhibez generation policy; fail closed.") from exc
    checks = {
        "SonicVhibez channel": policy.get("channel") == "SonicVhibez",
        "AI wildlife reconstruction permission": policy.get("safety", {}).get("aiGeneratedWildlifeVisualsAllowed") is True,
        "generation enabled": policy.get("generation", {}).get("enabled") is True,
        "production disposition enabled": policy.get("productionDisposition", {}).get("enabled") is True,
        "explicit generation authorization": policy.get("productionDisposition", {}).get("noGenerationAuthorized") is False,
        "verified zero-cost GPU backend": policy.get("runtime", {}).get("zeroCostBackendVerification", {}).get("verified") is True,
        "verified exact model/checkpoint/VAE rights": policy.get("generation", {}).get("modelRightsVerification", {}).get("verified") is True,
        "commercial model-artifact approval": policy.get("licensing", {}).get("commercialUseApproved") is True,
        "passed mirror provenance review": policy.get("licensing", {}).get("mirrorProvenanceReview", {}).get("status") == "PASS",
    }
    blocked = [name for name, passed in checks.items() if not passed]
    if blocked:
        raise RuntimeError(
            "BLOCKED: SonicVhibez permits disclosed AI wildlife reconstructions, "
            "but generation remains disabled until every gate is verified: "
            + ", ".join(blocked)
        )
    return policy

require_generation_authorized()

import os, shutil, subprocess, sys, json, hashlib, platform
from pathlib import Path

def run(cmd, cwd=None, env=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

gpu = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
if gpu.returncode != 0:
    raise RuntimeError('No NVIDIA GPU detected. In Kaggle Settings, select GPU and rerun this cell.')
print(gpu.stdout[:4000])
try:
    import urllib.request
    urllib.request.urlopen('https://huggingface.co', timeout=15).read(64)
except Exception as exc:
    raise RuntimeError('Internet is unavailable. Enable Internet in Kaggle Settings and rerun.') from exc
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    run(['apt-get','update','-qq'])
    run(['apt-get','install','-y','--no-install-recommends','ffmpeg'])
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    raise RuntimeError('FFmpeg/ffprobe are required for output validation.')
print('GPU, Internet and FFmpeg checks passed. Python:', platform.python_version())


In [ ]:
raise RuntimeError("DISABLED_BY_OWNER_AI_WILDLIFE_VISUAL_POLICY: original creator-captured wildlife media is required; AI wildlife generation is prohibited.")

require_generation_authorized()

from pathlib import Path
import os, shutil, subprocess

WAN_GP_COMMIT = 'f4cfc1259215577be9a2ee280c41efbedfb4459a'
WAN_GP_URL = 'https://github.com/deepbeepmeep/WanGP.git'
WAN_ROOT = Path('/kaggle/working/WanGP').resolve()
SESSION_ROOT = Path('/kaggle/temp/WanGP-data').resolve()
CACHE_ROOT = Path('/kaggle/temp/WanGP-cache').resolve()
OUTPUT_ROOT = Path('/kaggle/working/WanGP-outputs').resolve()
for folder in (SESSION_ROOT, CACHE_ROOT, OUTPUT_ROOT): folder.mkdir(parents=True, exist_ok=True)

if not WAN_ROOT.exists():
    run(['git', 'clone', '--no-tags', WAN_GP_URL, str(WAN_ROOT)])
run(['git', 'fetch', '--no-tags', 'origin', WAN_GP_COMMIT], cwd=WAN_ROOT)
run(['git', 'checkout', '--detach', WAN_GP_COMMIT], cwd=WAN_ROOT)
actual = subprocess.check_output(['git','rev-parse','HEAD'], cwd=WAN_ROOT, text=True).strip()
if actual != WAN_GP_COMMIT: raise RuntimeError(f'WanGP pin mismatch: {actual}')

def attach_storage(repo_path, storage_path):
    repo_path, storage_path = Path(repo_path), Path(storage_path)
    storage_path.mkdir(parents=True, exist_ok=True)
    if repo_path.is_symlink():
        if repo_path.resolve() != storage_path.resolve(): raise RuntimeError(f'Unexpected symlink: {repo_path}')
        return
    if repo_path.exists():
        if not repo_path.is_dir(): raise RuntimeError(f'Expected directory: {repo_path}')
        for child in list(repo_path.iterdir()):
            dest = storage_path / child.name
            if dest.exists(): raise RuntimeError(f'Refusing to overwrite existing session file: {dest}')
            shutil.move(str(child), str(dest))
        repo_path.rmdir()
    repo_path.symlink_to(storage_path, target_is_directory=True)

attach_storage(WAN_ROOT / 'ckpts', SESSION_ROOT / 'ckpts')
attach_storage(WAN_ROOT / 'loras', SESSION_ROOT / 'loras')
attach_storage(WAN_ROOT / 'outputs', OUTPUT_ROOT)
(CACHE_ROOT / 'huggingface').mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT / 'huggingface' / 'hub')
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
os.environ['XDG_CACHE_HOME'] = str(CACHE_ROOT / '.cache')

license_path = WAN_ROOT / 'LICENSE.txt'
if not license_path.is_file(): raise RuntimeError('Pinned WanGP source is missing LICENSE.txt')
shutil.copy2(license_path, OUTPUT_ROOT / 'WanGP-LICENSE.txt')
print('Pinned WanGP commit:', actual)
print('Model cache:', SESSION_ROOT)
print('Generated outputs:', OUTPUT_ROOT)


## Install the pinned WanGP revision

The Git checkout is immutable at the commit above; this notebook never runs `git pull`. Python dependencies follow that commit's `requirements.txt` and are captured with `pip freeze` after installation. Kaggle's preinstalled CUDA PyTorch version is constrained so the dependency install does not silently replace it.


In [ ]:
raise RuntimeError("DISABLED_BY_OWNER_AI_WILDLIFE_VISUAL_POLICY: original creator-captured wildlife media is required; AI wildlife generation is prohibited.")

require_generation_authorized()

import importlib, os, subprocess, sys
env = os.environ.copy()
env['PIP_NO_CACHE_DIR'] = '1'
env['PIP_CACHE_DIR'] = str(CACHE_ROOT / 'pip')
import torch
if not torch.cuda.is_available(): raise RuntimeError('Kaggle PyTorch cannot access CUDA; restart with GPU enabled.')
constraints = CACHE_ROOT / 'torch-constraints.txt'
versions = {'torch': torch.__version__.split('+')[0]}
for name in ('torchvision','torchaudio'):
    try:
        mod = importlib.import_module(name)
        versions[name] = mod.__version__.split('+')[0]
    except Exception: pass
constraints.write_text(''.join(f'{k}=={v}\n' for k,v in versions.items()))
run([sys.executable,'-m','pip','install','--no-cache-dir','-r',str(WAN_ROOT / 'requirements.txt'),'-c',str(constraints)], env=env)
run([sys.executable,'-m','pip','freeze'], env=env)
freeze = subprocess.check_output([sys.executable,'-m','pip','freeze'], text=True, env=env)
(OUTPUT_ROOT / 'pip-freeze.txt').write_text(freeze)
print('WanGP dependencies installed. CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))


## Pin the exact test checkpoint and VAE

Checkpoint: `DeepBeepMeep/Wan2.2/wan2.2_text2video_5B_quanto_mbf16_int8.safetensors` at revision `16bb1ad…`, SHA-256 verified below. VAE: `Wan2.2_VAE.safetensors` at revision `4cb9607…`, SHA-256 verified below. These are pinned file revisions, not floating `main` downloads. The WanGP model definition created below has **no LoRA**.


In [ ]:
raise RuntimeError("DISABLED_BY_OWNER_AI_WILDLIFE_VISUAL_POLICY: original creator-captured wildlife media is required; AI wildlife generation is prohibited.")

require_generation_authorized()

from huggingface_hub import hf_hub_download
import hashlib, shutil

MODEL_REPO = 'DeepBeepMeep/Wan2.2'
MODEL_REVISION = '16bb1ad02fe4ca8319b67261397ea4eeaa4896c4'
MODEL_FILE = 'wan2.2_text2video_5B_quanto_mbf16_int8.safetensors'
MODEL_SHA256 = '463076f7625d8c7fc8912011cf5d02db7f0a8ae666e1758cd835e5a8517ef0e0'
VAE_REVISION = '4cb9607e5532be42f37a05778acfc47aee7701e1'
VAE_FILE = 'Wan2.2_VAE.safetensors'
VAE_SHA256 = 'bb6e5e515ca6af2cb2b7bbf8bfc790b8c306832c34e0a76193507c3739ebf54a'

def sha256(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

model_download = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, revision=MODEL_REVISION, cache_dir=str(CACHE_ROOT / 'hf-artifacts'))
if sha256(model_download) != MODEL_SHA256: raise RuntimeError('Model checkpoint SHA-256 mismatch; refusing to use it.')
model_target = SESSION_ROOT / 'ckpts' / MODEL_FILE
shutil.copy2(model_download, model_target)
vae_download = hf_hub_download(repo_id=MODEL_REPO, filename=VAE_FILE, revision=VAE_REVISION, cache_dir=str(CACHE_ROOT / 'hf-artifacts'))
if sha256(vae_download) != VAE_SHA256: raise RuntimeError('VAE SHA-256 mismatch; refusing to use it.')
vae_target = SESSION_ROOT / 'ckpts' / VAE_FILE
shutil.copy2(vae_download, vae_target)

assert sha256(model_target) == MODEL_SHA256
assert sha256(vae_target) == VAE_SHA256
print('Pinned model checkpoint verified:', model_target, model_target.stat().st_size)
print('Pinned VAE verified:', vae_target, vae_target.stat().st_size)


In [ ]:
raise RuntimeError("DISABLED_BY_OWNER_AI_WILDLIFE_VISUAL_POLICY: original creator-captured wildlife media is required; AI wildlife generation is prohibited.")

require_generation_authorized()

import json

FINETUNE_DIR = WAN_ROOT / 'finetunes'
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DEF = {
  'model': {
    'name': 'SonicVhibez Wan 2.2 TI2V 5B — pinned no-LoRA test',
    'architecture': 'ti2v_2_2',
    'description': 'Pinned 5B Wan2.2 text-to-video test profile. No FastWan or other LoRA is applied.',
    'infos': 'Internal evaluation only. Checkpoint mirror provenance/licence remains under review; commercial release is blocked. Base model card: https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B',
    'URLs': ['https://huggingface.co/' + MODEL_REPO + '/resolve/' + MODEL_REVISION + '/' + MODEL_FILE],
    'settings_dir': ['wan2_2_5B'],
    'group': 'wan2_2',
    'loras': [],
    'loras_multipliers': []
  },
  'video_length': 121,
  'guidance_scale': 5,
  'flow_shift': 5,
  'num_inference_steps': 30,
  'resolution': '832x480'
}
finetune_path = FINETUNE_DIR / 'sonicvhibez_wan22_ti2v5b_pinned.json'
finetune_path.write_text(json.dumps(MODEL_DEF, indent=2) + '\n')
print('Created custom no-LoRA WanGP model profile:', finetune_path)


## Launch WanGP and generate the test clip

Run the next cell and open the temporary Gradio link printed by WanGP. In the model selector choose **SonicVhibez Wan 2.2 TI2V 5B — pinned no-LoRA test** (refresh the model list if needed). Confirm the LoRA list is empty. Use the prompt below, 832×480, 121 frames, 30 steps, guidance 5, flow shift 5. Record the exact seed shown by WanGP.

**Test prompt:** An ultra-photorealistic wildlife documentary shot of an adult African elephant walking slowly through dry golden savannah grass at sunrise, its young calf walking close beside it; natural animal anatomy and proportions, subtle ear and trunk movement, realistic weight and foot placement, gentle dust in backlight, long-lens field-documentary cinematography, stable camera, natural colour and exposure, continuous physically plausible motion, no humans, no fences, no text, no logos. This is a synthetic illustrative scene, not a claim about a real recorded event.

**Negative prompt:** cartoon, anime, illustration, toy, plastic CGI, fantasy, extra legs, malformed trunk, duplicated animals, changing species, morphing body, sliding feet, impossible motion, flicker, jitter, warped background, text, watermark, logo, oversharpening, oversaturated colours.

Keep the launch cell running only while generating. Do not publish or schedule this test.


In [ ]:
raise RuntimeError("DISABLED_BY_OWNER_AI_WILDLIFE_VISUAL_POLICY: original creator-captured wildlife media is required; AI wildlife generation is prohibited.")

require_generation_authorized()

import os, subprocess, sys
env = os.environ.copy()
env['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT / 'huggingface' / 'hub')
env['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
env['XDG_CACHE_HOME'] = str(CACHE_ROOT / '.cache')
print('Launching pinned WanGP. Do not enter credentials or private information in the temporary public UI.')
print('When finished, stop this cell with the notebook Stop button.')
run([sys.executable, '-u', 'wgp.py', '--listen', '--server-port', '7860', '--share', '--profile', '5'], cwd=WAN_ROOT, env=env)


## Package the generated MP4 and provenance evidence

After generating and saving the clip in the WanGP gallery, run this cell. It selects the newest MP4 in the project output directory, checks the complete video decode, asks for the exact seed from the UI, and writes a ZIP containing the MP4, manifest, pinned source revision, licence copy and Python environment. The manifest deliberately records commercial-use status as HOLD and human review as PENDING.


In [ ]:
raise RuntimeError("DISABLED_BY_OWNER_AI_WILDLIFE_VISUAL_POLICY: original creator-captured wildlife media is required; AI wildlife generation is prohibited.")

require_generation_authorized()

import json, subprocess, hashlib, shutil, zipfile, time
from pathlib import Path
from IPython.display import Video, display

candidates = sorted([p for p in OUTPUT_ROOT.rglob('*.mp4') if p.is_file()], key=lambda p:p.stat().st_mtime, reverse=True)
if not candidates: raise RuntimeError(f'No MP4 found under {OUTPUT_ROOT}. Generate a clip in WanGP and save it to the gallery/output folder first.')
print('Latest MP4 candidates:')
for p in candidates[:10]: print(p, p.stat().st_size, 'bytes')
video_path = candidates[0]
override = input('Press Enter to use the newest MP4, or paste a different full MP4 path: ').strip()
if override: video_path = Path(override).resolve()
if not video_path.is_file() or video_path.suffix.lower() != '.mp4': raise RuntimeError('Selected output is not a readable MP4 file.')
seed_text = input('Enter the exact numeric seed shown in WanGP: ').strip()
session_ref = input('Enter the Kaggle notebook version/session reference shown in Kaggle: ').strip()
if not session_ref: raise RuntimeError('A real Kaggle notebook version/session reference is required; do not use a placeholder.')
if not seed_text.isdigit(): raise RuntimeError('An exact numeric seed is required; do not guess it.')

TEST_PROMPT = 'An ultra-photorealistic wildlife documentary shot of an adult African elephant walking slowly through dry golden savannah grass at sunrise, its young calf walking close beside it; natural animal anatomy and proportions, subtle ear and trunk movement, realistic weight and foot placement, gentle dust in backlight, long-lens field-documentary cinematography, stable camera, natural colour and exposure, continuous physically plausible motion, no humans, no fences, no text, no logos. This is a synthetic illustrative scene, not a claim about a real recorded event.'
NEGATIVE_PROMPT = 'cartoon, anime, illustration, toy, plastic CGI, fantasy, extra legs, malformed trunk, duplicated animals, changing species, morphing body, sliding feet, impossible motion, flicker, jitter, warped background, text, watermark, logo, oversharpening, oversaturated colours'
probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration:stream=codec_type,width,height,codec_name,avg_frame_rate','-of','json',str(video_path)], text=True, capture_output=True)
if probe.returncode: raise RuntimeError('ffprobe failed: ' + probe.stderr[-2000:])
probe_data = json.loads(probe.stdout)
video_stream = next((s for s in probe_data.get('streams',[]) if s.get('codec_type')=='video'), None)
if not video_stream: raise RuntimeError('MP4 has no video stream.')
duration = float(probe_data.get('format',{}).get('duration',0))
if duration < 1 or duration > 15: raise RuntimeError(f'Test shot duration outside 1–15s: {duration}')
decode = subprocess.run(['ffmpeg','-v','error','-xerror','-i',str(video_path),'-f','null','-'], text=True, capture_output=True)
if decode.returncode: raise RuntimeError('Full MP4 decode failed: ' + decode.stderr[-2000:])

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

checkpoint_file = SESSION_ROOT / 'ckpts' / MODEL_FILE
vae_file = SESSION_ROOT / 'ckpts' / VAE_FILE
manifest = {
  'schemaVersion': 1,
  'provider': 'wangp_kaggle',
  'channel': 'SonicVhibez',
  'status': 'GENERATED_TEST_UNREVIEWED',
  'generatedAtUtc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
  'kaggleSessionReference': session_ref,
  'wanGpRepository': WAN_GP_URL,
  'wanGpCommit': WAN_GP_COMMIT,
  'modelId': 'Wan 2.2 TI2V 5B (WanGP custom finetune, no LoRA)',
  'modelRepository': MODEL_REPO,
  'modelRevision': MODEL_REVISION,
  'checkpointFilename': MODEL_FILE,
  'checkpointSha256': sha256_file(checkpoint_file),
  'checkpointRepository': MODEL_REPO,
  'checkpointExpectedSha256': MODEL_SHA256,
  'checkpointLicenseStatus': 'HOLD_UNVERIFIED_MIRROR_DERIVATIVE_PROVENANCE',
  'checkpointLicenseUrl': 'https://huggingface.co/DeepBeepMeep/Wan2.2/blob/'+MODEL_REVISION+'/'+MODEL_FILE,
  'upstreamBaseModel': 'Wan-AI/Wan2.2-TI2V-5B',
  'upstreamBaseModelRevision': '921dbaf3f1674a56f47e83fb80a34bac8a8f203e',
  'upstreamBaseModelLicense': 'Apache-2.0',
  'upstreamBaseModelLicenseUrl': 'https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B',
  'vaeFilename': VAE_FILE,
  'vaeRevision': VAE_REVISION,
  'vaeSha256': sha256_file(vae_file),
  'modelAssetInventory': [{'path':str(p.relative_to(SESSION_ROOT / 'ckpts')),'sha256':sha256_file(p),'bytes':p.stat().st_size} for p in sorted((SESSION_ROOT / 'ckpts').rglob('*')) if p.is_file()],
  'loras': [],
  'prompt': TEST_PROMPT,
  'negativePrompt': NEGATIVE_PROMPT,
  'seed': int(seed_text),
  'generationSettings': {'resolution':'832x480','frames':121,'targetDurationSeconds':5.04,'steps':30,'guidanceScale':5,'flowShift':5,'aspectRatio':'16:9'},
  'videoFilename': 'wildlife-test.mp4',
  'sourceVideoFilename': video_path.name,
  'videoSha256': sha256_file(video_path),
  'videoDurationSeconds': duration,
  'videoWidth': int(video_stream.get('width',0)),
  'videoHeight': int(video_stream.get('height',0)),
  'videoCodec': video_stream.get('codec_name'),
  'ffprobe': probe_data,
  'fullDecodePassed': True,
  'humanVisualReviewStatus': 'PENDING',
  'commercialUseApproved': False,
  'commercialUseStatus': 'HOLD_PENDING_CHECKPOINT_PROVENANCE_AND_LICENCE_CONFIRMATION',
  'publishingStatus': 'NOT_APPROVED'
}
if manifest['checkpointSha256'] != MODEL_SHA256 or manifest['vaeSha256'] != VAE_SHA256: raise RuntimeError('Model/VAE file hash changed after initial verification.')
package_dir = Path('/kaggle/working/sonicvhibez-wangp-test-package')
if package_dir.exists(): shutil.rmtree(package_dir)
package_dir.mkdir(parents=True)
shutil.copy2(video_path, package_dir / 'wildlife-test.mp4')
(package_dir / 'generation-manifest.json').write_text(json.dumps(manifest, indent=2) + '\
')
(package_dir / 'wanGp-commit.txt').write_text(WAN_GP_COMMIT+'\
')
shutil.copy2(WAN_ROOT / 'LICENSE.txt', package_dir / 'WanGP-LICENSE.txt')
shutil.copy2(OUTPUT_ROOT / 'pip-freeze.txt', package_dir / 'pip-freeze.txt')
zip_path = Path('/kaggle/working/sonicvhibez-wangp-test-package.zip')
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in package_dir.iterdir(): z.write(f, arcname=f.name)
print(json.dumps({k:manifest[k] for k in ('videoFilename','videoSha256','videoDurationSeconds','videoWidth','videoHeight','checkpointSha256','wanGpCommit','commercialUseStatus','humanVisualReviewStatus')}, indent=2))
display(Video(str(video_path), embed=True))
print('Package ready:', zip_path, 'size:', zip_path.stat().st_size)


## What this run does not approve

A successful FFmpeg decode proves only that the file is technically readable. It does not prove photorealism, animal anatomy, factual accuracy, source/model rights, narration rights, YouTube disclosure compliance or human approval. The generated shot is synthetic and must never be described as a real wildlife encounter. The checkpoint licence/provenance hold must be resolved before commercial use. Do not schedule or publish this test clip.
